# ResNet293 PyTorch POC

Minimal proof-of-concept for loading the WeSpeaker ResNet293 PyTorch checkpoint, extracting 80-bin filterbank features from one audio file, and producing a 256-dimensional embedding.


In [1]:
import sys
import types
from pathlib import Path
import importlib.util

import yaml
import torch
import soundfile as sf
import torchaudio
import torchaudio.compliance.kaldi as kaldi


In [2]:
# ===== Config =====
REPO_ROOT = Path('/home/SpeakerRec/BioVoice')
MODEL_DIR = REPO_ROOT / 'data' / 'models' / 'wespeaker_resnet293_lm'
AUDIO_PATH = REPO_ROOT / 'data' / 'wavs' / 'eden_001.wav'
WESPEAKER_MODELS_DIR = REPO_ROOT / 'external' / 'wespeaker' / 'wespeaker' / 'models'

CONFIG_PATH = MODEL_DIR / 'config.yaml'
CKPT_PATH = MODEL_DIR / 'avg_model.pt'
TARGET_SR = 16000

print('CONFIG_PATH =', CONFIG_PATH)
print('CONFIG EXISTS =', CONFIG_PATH.exists())
print('CKPT_PATH =', CKPT_PATH)
print('CKPT EXISTS =', CKPT_PATH.exists())
print('AUDIO_PATH =', AUDIO_PATH)
print('AUDIO EXISTS =', AUDIO_PATH.exists())
print('WESPEAKER_MODELS_DIR =', WESPEAKER_MODELS_DIR)
print('WESPEAKER_MODELS_DIR EXISTS =', WESPEAKER_MODELS_DIR.exists())


CONFIG_PATH = /home/SpeakerRec/BioVoice/data/models/wespeaker_resnet293_lm/config.yaml
CONFIG EXISTS = True
CKPT_PATH = /home/SpeakerRec/BioVoice/data/models/wespeaker_resnet293_lm/avg_model.pt
CKPT EXISTS = True
AUDIO_PATH = /home/SpeakerRec/BioVoice/data/wavs/eden_001.wav
AUDIO EXISTS = True
WESPEAKER_MODELS_DIR = /home/SpeakerRec/BioVoice/external/wespeaker/wespeaker/models
WESPEAKER_MODELS_DIR EXISTS = True


In [3]:
# Load only the exact WeSpeaker model files we need, without importing the full package.
wespeaker_pkg = types.ModuleType('wespeaker')
wespeaker_models_pkg = types.ModuleType('wespeaker.models')
sys.modules['wespeaker'] = wespeaker_pkg
sys.modules['wespeaker.models'] = wespeaker_models_pkg

pool_spec = importlib.util.spec_from_file_location(
    'wespeaker.models.pooling_layers',
    str(WESPEAKER_MODELS_DIR / 'pooling_layers.py'),
)
pool_mod = importlib.util.module_from_spec(pool_spec)
sys.modules['wespeaker.models.pooling_layers'] = pool_mod
pool_spec.loader.exec_module(pool_mod)

resnet_spec = importlib.util.spec_from_file_location(
    'wespeaker.models.resnet',
    str(WESPEAKER_MODELS_DIR / 'resnet.py'),
)
resnet_mod = importlib.util.module_from_spec(resnet_spec)
sys.modules['wespeaker.models.resnet'] = resnet_mod
resnet_spec.loader.exec_module(resnet_mod)

ResNet293 = resnet_mod.ResNet293
print('Loaded ResNet293 constructor from direct file import.')


Loaded ResNet293 constructor from direct file import.


In [4]:
with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    configs = yaml.safe_load(f)

print('model =', configs['model'])
print('model_args =', configs['model_args'])
print('fbank_args =', configs['dataset_args']['fbank_args'])

model = ResNet293(**configs['model_args'])

state = torch.load(CKPT_PATH, map_location='cpu')
print('checkpoint type =', type(state))
if isinstance(state, dict):
    print('checkpoint keys sample =', list(state.keys())[:20])
    if 'state_dict' in state:
        state = state['state_dict']
    elif 'model' in state:
        state = state['model']

if isinstance(state, dict):
    cleaned = {}
    for k, v in state.items():
        nk = k[7:] if k.startswith('module.') else k
        cleaned[nk] = v
    state = cleaned

missing, unexpected = model.load_state_dict(state, strict=False)
print('missing keys =', missing)
print('unexpected keys =', unexpected)

model.eval()


model = ResNet293
model_args = {'embed_dim': 256, 'feat_dim': 80, 'pooling_func': 'TSTP', 'two_emb_layer': False}
fbank_args = {'dither': 1.0, 'frame_length': 25, 'frame_shift': 10, 'num_mel_bins': 80}
checkpoint type = <class 'collections.OrderedDict'>
checkpoint keys sample = ['conv1.weight', 'bn1.weight', 'bn1.bias', 'bn1.running_mean', 'bn1.running_var', 'bn1.num_batches_tracked', 'layer1.0.conv1.weight', 'layer1.0.bn1.weight', 'layer1.0.bn1.bias', 'layer1.0.bn1.running_mean', 'layer1.0.bn1.running_var', 'layer1.0.bn1.num_batches_tracked', 'layer1.0.conv2.weight', 'layer1.0.bn2.weight', 'layer1.0.bn2.bias', 'layer1.0.bn2.running_mean', 'layer1.0.bn2.running_var', 'layer1.0.bn2.num_batches_tracked', 'layer1.0.conv3.weight', 'layer1.0.bn3.weight']
missing keys = []
unexpected keys = ['projection.weight']


ResNet(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential(
        (0): Conv2d(32, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (1): Bottleneck(
      (

In [5]:
waveform_np, sr = sf.read(str(AUDIO_PATH))
print('loaded waveform raw shape =', waveform_np.shape, '| sample_rate =', sr)

if waveform_np.ndim == 1:
    waveform_np = waveform_np[None, :]
else:
    waveform_np = waveform_np.T

waveform = torch.from_numpy(waveform_np).float()
if sr != TARGET_SR:
    waveform = torchaudio.functional.resample(waveform, sr, TARGET_SR)
    sr = TARGET_SR

waveform = waveform[:1, :]
waveform = waveform * (1 << 15)

print('waveform used for fbank shape =', tuple(waveform.shape), '| sample_rate =', sr)


loaded waveform raw shape = (37797,) | sample_rate = 16000
waveform used for fbank shape = (1, 37797) | sample_rate = 16000


In [6]:
fbank_args = configs['dataset_args']['fbank_args']
feats = kaldi.fbank(
    waveform,
    num_mel_bins=fbank_args['num_mel_bins'],
    frame_length=fbank_args['frame_length'],
    frame_shift=fbank_args['frame_shift'],
    dither=0.0,
    sample_frequency=TARGET_SR,
)

print('fbank shape =', tuple(feats.shape))
display(feats[:5])


fbank shape = (234, 80)


tensor([[ 5.6752,  6.0484,  6.3748,  6.9200,  7.9883,  8.5381,  8.4804,  8.0205,
          7.9255,  7.4939,  7.2241,  7.3500,  8.1983,  8.0522,  6.9999,  6.0181,
          5.2545,  6.6669,  7.5251,  7.4321,  7.7316,  8.5028,  7.9306,  5.7850,
          6.3817,  7.6837,  7.8684,  7.8292,  7.0667,  9.0209,  9.9456,  9.6913,
          7.9224,  8.6748,  9.3543,  7.5938,  8.5446,  8.9407,  9.0444,  9.4644,
          8.7396,  8.2402,  9.4476,  8.5060,  8.8384,  8.2314,  8.8910, 10.3882,
         10.3363,  9.1236,  9.0837,  8.9763,  9.1546,  9.2415,  8.8134,  9.3734,
          9.6670,  8.3396,  9.7441,  9.8086,  8.7052,  9.2764,  9.2134,  8.4902,
          8.7673, 10.1426,  9.4583,  8.8884,  8.6196,  9.0092,  8.5793,  9.5740,
          9.5219,  9.4425,  9.5446,  8.3469,  8.1354,  8.4687,  7.9565,  7.3543],
        [ 7.2703,  8.7437,  9.3083,  8.5273,  8.2263,  8.4279,  8.8917,  8.8856,
          9.0587,  8.0911,  5.7082,  5.9061,  6.1982,  6.5389,  7.5768,  7.1840,
          5.1811,  4.5218, 

In [7]:
feats = feats.unsqueeze(0)
with torch.no_grad():
    outputs = model(feats)
    emb = outputs[-1] if isinstance(outputs, tuple) else outputs

print('feats shape =', tuple(feats.shape))
print('embedding shape =', tuple(emb.shape))
print('embedding dtype =', emb.dtype)
print('first 10 values =', emb[0, :10])


feats shape = (1, 234, 80)
embedding shape = (1, 256)
embedding dtype = torch.float32
first 10 values = tensor([ 0.0172, -0.1383,  0.1308,  0.0730,  0.1149,  0.1743, -0.0011, -0.2975,
         0.2450, -0.2618])
